Housing Tenure & Deprivation: Data Cleaning & Preprocessing Pipeline

Census Data 2011: QS405EW, QS412EW, QS119EW, KS601EW\
Census Data 2021: TS054, TS011\
Geography:   Local Authority Districts, England & Wales

In [15]:
# Install required engines
import subprocess
subprocess.run(["pip", "install", "xlrd", "openpyxl", "-q"], check=True)
print("xlrd and openpyxl ready")

xlrd and openpyxl ready


In [16]:
import pandas as pd
import numpy as np
import os

PATHS = {
    # 2011 tables (.xls workbooks — use Numbers sheet)
    "qs405ew": "/content/2011 Tenure breakdown QS405EW .xls",
    "qs412ew": "/content/2011 Overcrowding QS412EW .xls",
    "qs119ew": "/content/2011 Deprivation dimensions QS119EW .xls",
    "ks601ew": "/content/2011 Economic activity KS601EW .xls",

    # 2021 tables (.csv bulk downloads — ltla geography level)
    "ts054":   "/content/census2021-ts054-ltla.csv",
    "ts011":   "/content/census2021-ts011-ltla.csv",
}

In [17]:
OUTPUT_DIR = "data/processed/"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def load_and_inspect(path, label):
    """
    Load data from either:
      - .xls / .xlsx workbook (2011 tables): reads the Numbers sheet,
        auto-detects the real header row past Nomis metadata rows.
      - .csv file (2021 bulk downloads): reads directly with pandas.
    Always uses raw counts (not percentages) so boundary mergers work
    correctly by summing before recomputing percentages.
    """
    print(f"\n{'='*60}")
    print(f"Loading: {label}")
    print(f"{'='*60}")

    ext = os.path.splitext(path)[1].lower()

    if ext == ".csv":
        df = pd.read_csv(path)
        print(f"  Format          : CSV")
    else:
        engine = "xlrd" if ext == ".xls" else "openpyxl"
        xl = pd.ExcelFile(path, engine=engine)
        print(f"  Sheets found    : {xl.sheet_names}")

        # Target Numbers sheet
        preferred = ["Numbers", "numbers", "Data", "data", "Sheet1"]
        target_sheet = next(
            (s for s in xl.sheet_names if any(p in s for p in preferred)),
            xl.sheet_names[0]
)
        print(f"  Reading sheet   : '{target_sheet}'")

        # Find real header row by scanning for geography-related keywords
        temp_df = pd.read_excel(path, sheet_name=target_sheet, engine=engine, header=None)
        header_keywords = ["geography code", "mnemonic", "code", "area", "local authority"]
        header_row_index = 0
        for i in range(min(20, len(temp_df))):
            row_values = [str(v).lower().strip() for v in temp_df.iloc[i].dropna()]
            if any(any(kw in v for kw in header_keywords) for v in row_values):
                header_row_index = i
                print(f"  Header at row   : {i}")
                break

        df = pd.read_excel(path, sheet_name=target_sheet, engine=engine, header=header_row_index)
        df = df.dropna(how="all").reset_index(drop=True)

    print(f"  Shape           : {df.shape[0]} rows x {df.shape[1]} cols")
    print(f"  Columns         : {list(df.columns)}")
    print(f"  Missing vals    : {df.isnull().sum().sum()} total")
    print(f"\n  First 3 rows:\n{df.head(3)}")
    return df


def check_missing(df, label):
    """Report missing values per column."""
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f"  [{label}] No missing values found.")
    else:
        print(f"  [{label}] Missing values detected:")
        print(missing)
    return missing


def standardise_geo_col(df, possible_names=("geography code", "mnemonic",
                                             "geo_code", "geography_code", "code")):
    cols_lower = {c.lower().strip(): c for c in df.columns}
    for name in possible_names:
        if name in cols_lower:
            df = df.rename(columns={cols_lower[name]: "geo_code"})
            print(f"  Renamed '{cols_lower[name]}' → 'geo_code'")
            return df

    # Fallback: detect by content pattern (LAD codes start E06-E09 or W06)
    geo_code_pattern = r"^(E0[6-9]|W06)[0-9]{6}"
    for col_name in df.columns:
        if pd.api.types.is_numeric_dtype(df[col_name]):
            continue
        non_null = df[col_name].dropna().astype(str)
        if non_null.empty:
            continue
        if non_null.str.match(geo_code_pattern).mean() > 0.8:
            df = df.rename(columns={col_name: "geo_code"})
            print(f"  Detected and renamed '{col_name}' → 'geo_code' by content pattern")
            return df

    raise KeyError(
        f"Could not find a geography code column. "
        f"Available columns: {list(df.columns)}"
    )


def standardise_name_col(df, possible_names=("geography", "geography name",
                                              "area name", "area_name",
                                              "local authority", "region", "country")):
    cols_lower = {c.lower().strip(): c for c in df.columns}
    for name in possible_names:
        if name in cols_lower:
            df = df.rename(columns={cols_lower[name]: "area_name"})
            print(f"  Renamed '{cols_lower[name]}' → 'area_name'")
            return df

    # Fallback: substring match
    for original_col in df.columns:
        col_lower = original_col.lower().strip()
        if any(kw in col_lower for kw in ["geography", "area", "local authority",
                                           "country", "region", "borough"]):
            df = df.rename(columns={original_col: "area_name"})
            print(f"  Detected and renamed '{original_col}' → 'area_name' by keyword")
            return df

    print("  WARNING: Could not find an area name column.")
    return df


def clean_geo_code(df):
    """Strip whitespace, keep only LAD-level codes (E06-E09, W06)."""
    df["geo_code"] = df["geo_code"].astype(str).str.strip()
    before = len(df)
    df = df[df["geo_code"].str.match(r"^(E0[6-9]|W06)")]
    after = len(df)
    print(f"  Dropped {before - after} non-LAD rows ({before} → {after})")
    return df


def convert_to_numeric(df, exclude=("geo_code", "area_name")):
    """Convert all non-identifier columns to numeric, coercing errors."""
    for col in df.columns:
        if col not in exclude:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def compute_percentages(df, total_col, exclude=("geo_code", "area_name")):
    """Add pct_ columns for every raw count column."""
    for col in df.columns:
        if col not in exclude and col != total_col and not col.startswith("pct_"):
            df[f"pct_{col}"] = (df[col] / df[total_col] * 100).round(2)
    return df


# Load all raw files
raw = {}
for key, path in PATHS.items():
    raw[key] = load_and_inspect(path, key.upper())



Loading: QS405EW
  Sheets found    : ['Information', 'QS405EW_Numbers', 'QS405EW_Percentages', 'QS405EW_Ranks', 'Metadata']
  Reading sheet   : 'QS405EW_Numbers'
  Header at row   : 10
  Shape           : 402 rows x 18 cols
  Columns         : ['Area code', 'Area name', 'Unnamed: 2', 'Unnamed: 3', 'All categories: Tenure', 'Owned: Total', 'Owned: Owned outright', 'Owned: Owned with a mortgage or loan', 'Shared ownership (part owned and part rented)', 'Social rented: Total', 'Social rented: Rented from council (Local Authority)', 'Social rented: Other social rented', 'Private rented: Total', 'Private rented: Private landlord or letting agency', 'Private rented: Employer of a household member', 'Private rented: Relative or friend of household member', 'Private rented: Other', 'Living rent free']
  Missing vals    : 883 total

  First 3 rows:
   Area code          Area name Unnamed: 2 Unnamed: 3 All categories: Tenure  \
0        NaN                NaN        NaN        NaN             H

CLEAN 2011: QS405EW — TENURE (HOUSEHOLDS)

In [18]:
print("\n\n>>> CLEANING QS405EW — 2011 Tenure")
t11 = raw["qs405ew"].copy()
t11 = standardise_geo_col(t11)
t11 = standardise_name_col(t11)
t11 = clean_geo_code(t11)
t11 = convert_to_numeric(t11)

rename_tenure_2011 = {
    "All households": "total_households",
    "Owned: Owned outright": "owned_outright",
    "Owned: Owned with a mortgage or loan": "owned_mortgage",
    "Shared ownership (part owned and part rented)": "shared_ownership",
    "Social rented: Rented from council (Local Authority)": "social_council",
    "Social rented: Other social rented": "social_other",
    "Private rented: Private landlord or letting agency": "private_rented",
    "Private rented: Other private rented": "private_other",
    "Living rent free": "rent_free",
}
t11.rename(columns={k: v for k, v in rename_tenure_2011.items()
                    if k in t11.columns}, inplace=True)

# Create combined useful columns
if "social_council" in t11.columns and "social_other" in t11.columns:
    t11["social_rented_total"] = t11["social_council"] + t11["social_other"]
if "private_rented" in t11.columns and "private_other" in t11.columns:
    t11["private_rented_total"] = t11["private_rented"] + t11["private_other"]
if "owned_outright" in t11.columns and "owned_mortgage" in t11.columns:
    t11["owned_total"] = t11["owned_outright"] + t11["owned_mortgage"]

# Add percentage columns
if "total_households" in t11.columns:
    t11 = compute_percentages(t11, "total_households")

check_missing(t11, "QS405EW")
print(f"  Final shape: {t11.shape}")



>>> CLEANING QS405EW — 2011 Tenure
  Detected and renamed 'Area code' → 'geo_code' by content pattern
  Renamed 'Area name' → 'area_name'
  Dropped 54 non-LAD rows (402 → 348)
  [QS405EW] Missing values detected:
area_name     348
Unnamed: 2    348
Unnamed: 3    348
dtype: int64
  Final shape: (348, 20)


CLEAN 2011: QS412EW — OCCUPANCY RATING (BEDROOMS)

In [19]:
print("\n\n>>> CLEANING QS412EW — 2011 Occupancy Rating (Bedrooms)")
o11 = raw["qs412ew"].copy()
o11 = standardise_geo_col(o11)
o11 = standardise_name_col(o11)
o11 = clean_geo_code(o11)
o11 = convert_to_numeric(o11)

rename_occ_2011 = {
    "All households": "total_households",
    "Occupancy rating of bedrooms: +2 or more": "occ_plus2_more",
    "Occupancy rating of bedrooms: +1": "occ_plus1",
    "Occupancy rating of bedrooms: 0": "occ_zero",
    "Occupancy rating of bedrooms: -1": "occ_minus1",
    "Occupancy rating of bedrooms: -2 or less": "occ_minus2_less",
}
o11.rename(columns={k: v for k, v in rename_occ_2011.items()
                    if k in o11.columns}, inplace=True)

# Overcrowded = any negative occupancy rating
neg_cols = [c for c in ["occ_minus1", "occ_minus2_less"] if c in o11.columns]
if neg_cols:
    o11["overcrowded_total"] = o11[neg_cols].sum(axis=1)

if "total_households" in o11.columns:
    o11 = compute_percentages(o11, "total_households")

check_missing(o11, "QS412EW")
print(f"  Final shape: {o11.shape}")



>>> CLEANING QS412EW — 2011 Occupancy Rating (Bedrooms)
  Detected and renamed 'Area code' → 'geo_code' by content pattern
  Renamed 'Area name' → 'area_name'
  Dropped 53 non-LAD rows (401 → 348)
  [QS412EW] Missing values detected:
area_name     348
Unnamed: 2    348
Unnamed: 3    348
dtype: int64
  Final shape: (348, 10)


CLEAN 2011: QS119EW — DEPRIVATION DIMENSIONS

In [20]:
print("\n\n>>> CLEANING QS119EW — 2011 Deprivation Dimensions")
d11 = raw["qs119ew"].copy()
d11 = standardise_geo_col(d11)
d11 = standardise_name_col(d11)
d11 = clean_geo_code(d11)
d11 = convert_to_numeric(d11)

rename_dep_2011 = {
    "All households": "total_households",
    "Household is not deprived in any dimension": "deprived_0",
    "Household is deprived in 1 dimension": "deprived_1",
    "Household is deprived in 2 dimensions": "deprived_2",
    "Household is deprived in 3 dimensions": "deprived_3",
    "Household is deprived in 4 dimensions": "deprived_4",
}
d11.rename(columns={k: v for k, v in rename_dep_2011.items()
                    if k in d11.columns}, inplace=True)

# Multi-dimensional deprivation (2+ dimensions)
dep_cols = [c for c in ["deprived_2", "deprived_3", "deprived_4"]
            if c in d11.columns]
if dep_cols:
    d11["multi_deprived"] = d11[dep_cols].sum(axis=1)

# Any deprivation (1+ dimensions)
any_dep_cols = [c for c in ["deprived_1", "deprived_2", "deprived_3",
                             "deprived_4"] if c in d11.columns]
if any_dep_cols:
    d11["any_deprived"] = d11[any_dep_cols].sum(axis=1)

if "total_households" in d11.columns:
    d11 = compute_percentages(d11, "total_households")

check_missing(d11, "QS119EW")
print(f"  Final shape: {d11.shape}")



>>> CLEANING QS119EW — 2011 Deprivation Dimensions
  Detected and renamed 'Area code' → 'geo_code' by content pattern
  Renamed 'Area name' → 'area_name'
  Dropped 51 non-LAD rows (399 → 348)
  [QS119EW] Missing values detected:
area_name     348
Unnamed: 2    348
Unnamed: 3    348
dtype: int64
  Final shape: (348, 12)


CLEAN 2011: KS601EW — ECONOMIC ACTIVITY

In [21]:
print("\n\n>>> CLEANING KS601EW — 2011 Economic Activity")
e11 = raw["ks601ew"].copy()
e11 = standardise_geo_col(e11)
e11 = standardise_name_col(e11)
e11 = clean_geo_code(e11)
e11 = convert_to_numeric(e11)

rename_eco_2011 = {
    "All usual residents aged 16 to 74": "total_16_74",
    "Economically active: Total": "economically_active_total",
    "Economically active: Employed": "employed_total",
    "Economically active: Employed: Full-time": "employed_full_time",
    "Economically active: Employed: Part-time": "employed_part_time",
    "Economically active: Employed: Self-employed": "self_employed",
    "Economically active: Unemployed": "unemployed",
    "Economically inactive: Total": "inactive_total",
    "Economically inactive: Retired": "inactive_retired",
    "Economically inactive: Student (including full-time students)": "inactive_student",
}
e11.rename(columns={k: v for k, v in rename_eco_2011.items()
                    if k in e11.columns}, inplace=True)

# The required 'pct_unemployed', 'pct_employed_total', 'pct_inactive_total',
# 'pct_inactive_sick', 'pct_self_employed' are derived from these base columns.
# Ensure 'total_16_74' is used as the base for percentages as specified in boundary changes.

if "total_16_74" in e11.columns:
    e11 = compute_percentages(e11, "total_16_74")

check_missing(e11, "KS601EW")
print(f"  Final shape: {e11.shape}")



>>> CLEANING KS601EW — 2011 Economic Activity
  Detected and renamed 'Area code' → 'geo_code' by content pattern
  Renamed 'Area name' → 'area_name'
  Dropped 51 non-LAD rows (399 → 348)
  [KS601EW] Missing values detected:
area_name     348
Unnamed: 2    348
Unnamed: 3    348
dtype: int64
  Final shape: (348, 19)


CLEAN 2021: TS054 — TENURE

In [33]:
# ── Fix 2011 column names & recompute percentages ──────────────────

# Drop junk Unnamed columns
for _df in [t11, o11, d11, e11]:
    junk = [c for c in _df.columns if str(c).startswith("Unnamed")]
    _df.drop(columns=junk, inplace=True)

# ── t11 ──
t11 = t11.loc[:, ~t11.columns.duplicated()]
t11.rename(columns={
    "All categories: Tenure":   "total_households",
    "Owned: Total":             "owned_total",
    "Social rented: Total":     "social_rented_total",
    "Private rented: Total":    "private_rented_total",
}, inplace=True)
t11.drop(columns=[c for c in t11.columns if c.startswith("pct_")], inplace=True)
if "total_households" in t11.columns:
  t11 = t11.loc[:, ~t11.columns.duplicated(keep='first')]
  t11 = compute_percentages(t11, "total_households")


# ── o11 ──
o11 = o11.loc[:, ~o11.columns.duplicated()]
o11.rename(columns={
    "All categories: Occupancy rating (bedrooms) ": "total_households",
    "Occupancy rating (bedrooms) of +2 or more":    "occ_plus2_more",
    "Occupancy rating (bedrooms) of +1":            "occ_plus1",
    "Occupancy rating (bedrooms) of 0":             "occ_zero",
    "Occupancy rating (bedrooms) of -1":            "occ_minus1",
    "Occupancy rating (bedrooms) of -2 or less":    "occ_minus2_less",
}, inplace=True)
neg_cols = [c for c in ["occ_minus1", "occ_minus2_less"] if c in o11.columns]
if neg_cols:
    o11["overcrowded_total"] = o11[neg_cols].sum(axis=1)
o11.drop(columns=[c for c in o11.columns if c.startswith("pct_")], inplace=True)
if "total_households" in o11.columns:
  o11 = o11.loc[:, ~o11.columns.duplicated(keep='first')]
  o11 = compute_percentages(o11, "total_households")

# ── d11 ──
d11 = d11.loc[:, ~d11.columns.duplicated()]
d11.rename(columns={
    "All categories: Classification of household deprivation": "total_households",
}, inplace=True)
d11.drop(columns=[c for c in d11.columns if c.startswith("pct_")], inplace=True)
if "total_households" in d11.columns:
  d11 = d11.loc[:, ~d11.columns.duplicated(keep='first')]
  d11 = compute_percentages(d11, "total_households")

# ── e11 ──
e11 = e11.loc[:, ~e11.columns.duplicated()]
e11.rename(columns={
    "All categories: Economic activity":                   "total_16_74",
    "Economically active: Employee: Part-time":            "employed_parttime",
    "Economically active: Employee: Full-time":            "employed_fulltime",
    "Economically active: Self-employed":                  "self_employed",
    "Economically active: Full-time student":              "active_student",
    "Economically inactive: Looking after home or family": "inactive_home",
    "Economically inactive: Long-term sick or disabled":   "inactive_sick",
    "Economically inactive: Other":                        "inactive_other",
}, inplace=True)
emp_cols = [c for c in ["employed_fulltime", "employed_parttime", "self_employed"] if c in e11.columns]
if emp_cols:
    e11["employed_total"] = e11[emp_cols].sum(axis=1)
inactive_cols = [c for c in ["inactive_retired", "inactive_student", "inactive_home",
                               "inactive_sick", "inactive_other"] if c in e11.columns]
if inactive_cols:
    e11["inactive_total"] = e11[inactive_cols].sum(axis=1)
e11.drop(columns=[c for c in e11.columns if c.startswith("pct_")], inplace=True)
if "total_16_74" in e11.columns:
  e11 = e11.loc[:, ~e11.columns.duplicated(keep='first')]
  e11 = compute_percentages(e11, "total_16_74")

# Verify
print("t11 pct cols:", [c for c in t11.columns if c.startswith("pct_")])
print("o11 pct cols:", [c for c in o11.columns if c.startswith("pct_")])
print("d11 pct cols:", [c for c in d11.columns if c.startswith("pct_")])
print("e11 pct cols:", [c for c in e11.columns if c.startswith("pct_")])

t11 pct cols: ['pct_owned_total', 'pct_owned_outright', 'pct_owned_mortgage', 'pct_shared_ownership', 'pct_social_rented_total', 'pct_social_council', 'pct_social_other', 'pct_private_rented_total', 'pct_private_rented', 'pct_Private rented: Employer of a household member', 'pct_Private rented: Relative or friend of household member', 'pct_Private rented: Other', 'pct_rent_free']
o11 pct cols: ['pct_occ_plus2_more', 'pct_occ_plus1', 'pct_occ_zero', 'pct_occ_minus1', 'pct_occ_minus2_less', 'pct_overcrowded_total']
d11 pct cols: ['pct_deprived_0', 'pct_deprived_1', 'pct_deprived_2', 'pct_deprived_3', 'pct_deprived_4', 'pct_multi_deprived', 'pct_any_deprived']
e11 pct cols: ['pct_employed_parttime', 'pct_employed_fulltime', 'pct_self_employed', 'pct_unemployed', 'pct_active_student', 'pct_inactive_retired', 'pct_inactive_student', 'pct_inactive_home', 'pct_inactive_sick', 'pct_inactive_other', 'pct_Unemployed: Age 16 to 24', 'pct_Unemployed: Age 50 to 74', 'pct_Unemployed: Never worked', 

In [34]:
print("\n\n>>> CLEANING TS054 — 2021 Tenure")
t21 = raw["ts054"].copy()
t21 = standardise_geo_col(t21)
t21 = standardise_name_col(t21)
t21 = clean_geo_code(t21)
t21 = convert_to_numeric(t21)

rename_tenure_2021 = {
    "Tenure of household: Total: All households": "total_households",
    "Tenure of household: Owned": "owned_total",
    "Tenure of household: Owned: Owns outright": "owned_outright",
    "Tenure of household: Owned: Owns with a mortgage or loan": "owned_mortgage",
    "Tenure of household: Shared ownership": "shared_ownership",
    "Tenure of household: Social rented": "social_rented_total",
    "Tenure of household: Social rented: Rents from council or Local Authority": "social_council",
    "Tenure of household: Social rented: Other social rented": "social_other",
    "Tenure of household: Private rented": "private_rented_total",
    "Tenure of household: Private rented: Private landlord or letting agency": "private_rented",
    "Tenure of household: Private rented: Other private rented": "private_other",
    "Tenure of household: Lives rent free": "rent_free",
}
t21.rename(columns={k: v for k, v in rename_tenure_2021.items()
                    if k in t21.columns}, inplace=True)

if "total_households" in t21.columns:
    t21 = compute_percentages(t21, "total_households")

check_missing(t21, "TS054")
print(f"  Final shape: {t21.shape}")




>>> CLEANING TS054 — 2021 Tenure
  Renamed 'geography code' → 'geo_code'
  Renamed 'geography' → 'area_name'
  Dropped 0 non-LAD rows (331 → 331)
  [TS054] No missing values found.
  Final shape: (331, 29)


CLEAN 2021: TS011 — DEPRIVATION DIMENSIONS

In [35]:
print("\n\n>>> CLEANING TS011 — 2021 Deprivation Dimensions")
d21 = raw["ts011"].copy()
d21 = standardise_geo_col(d21)
d21 = standardise_name_col(d21)
d21 = clean_geo_code(d21)
d21 = convert_to_numeric(d21)

rename_dep_2021 = {
    "Household deprivation: Total: All households; measures: Value": "total_households",
    "Household deprivation: Household is not deprived in any dimension; measures: Value": "deprived_0",
    "Household deprivation: Household is deprived in one dimension; measures: Value": "deprived_1",
    "Household deprivation: Household is deprived in two dimensions; measures: Value": "deprived_2",
    "Household deprivation: Household is deprived in three dimensions; measures: Value": "deprived_3",
    "Household deprivation: Household is deprived in four dimensions; measures: Value": "deprived_4",
}
d21.rename(columns={k: v for k, v in rename_dep_2021.items()
                    if k in d21.columns}, inplace=True)

dep_cols_21 = [c for c in ["deprived_2", "deprived_3", "deprived_4"]
               if c in d21.columns]
if dep_cols_21:
    d21["multi_deprived"] = d21[dep_cols_21].sum(axis=1)

any_dep_cols_21 = [c for c in ["deprived_1", "deprived_2", "deprived_3",
                                "deprived_4"] if c in d21.columns]
if any_dep_cols_21:
    d21["any_deprived"] = d21[any_dep_cols_21].sum(axis=1)

if "total_households" in d21.columns:
    d21 = compute_percentages(d21, "total_households")

check_missing(d21, "TS011")
print(f"  Final shape: {d21.shape}")




>>> CLEANING TS011 — 2021 Deprivation Dimensions
  Renamed 'geography code' → 'geo_code'
  Renamed 'geography' → 'area_name'
  Dropped 0 non-LAD rows (331 → 331)
  [TS011] No missing values found.
  Final shape: (331, 19)


BOUNDARY HARMONISATION: ONS LAD (2011) to LAD (2021) Official Lookup

Downloaded directly from the ONS Open Geography Portal at runtime.\
This is the authoritative source for mapping 2011 LAD codes to their 2021 equivalents.\
CHGIND values: U = unchanged, M = merged, X = split & merged.\
We aggregate any 2011 LADs that map to the same 2021 LAD code,\
then recompute all percentages from the summed raw counts.

In [36]:
print("\n>>> LOADING ONS LAD 2011 → LAD 2021 LOOKUP FROM FILE")

LOOKUP_PATH = r"/content/Local_Authority_District_(2011)_to_Local_Authority_District_(2021)_Lookup_for_England_and_Wales.csv"

lookup_raw = pd.read_csv(LOOKUP_PATH, encoding="utf-8-sig")  # utf-8-sig strips the BOM character
print(f"  Loaded {len(lookup_raw)} rows")
print(f"  Columns: {list(lookup_raw.columns)}")
print(lookup_raw.head(5))



>>> LOADING ONS LAD 2011 → LAD 2021 LOOKUP FROM FILE
  Loaded 348 rows
  Columns: ['LAD11CD', 'LAD11NM', 'LAD11NMW', 'LAD21CD', 'LAD21NM', 'LAD21NMW', 'ObjectId']
     LAD11CD               LAD11NM LAD11NMW    LAD21CD               LAD21NM  \
0  E06000001            Hartlepool      NaN  E06000001            Hartlepool   
1  E06000002         Middlesbrough      NaN  E06000002         Middlesbrough   
2  E06000003  Redcar and Cleveland      NaN  E06000003  Redcar and Cleveland   
3  E06000004      Stockton-on-Tees      NaN  E06000004      Stockton-on-Tees   
4  E06000005            Darlington      NaN  E06000005            Darlington   

  LAD21NMW  ObjectId  
0      NaN         1  
1      NaN         2  
2      NaN         3  
3      NaN         4  
4      NaN         5  


In [37]:
def harmonise_boundaries(df, lookup, total_col, label):
    """
    Map 2011 LAD codes to 2021 LAD codes using the ONS official lookup,
    aggregate raw counts for any many-to-one mergers, then recompute
    all percentage columns from the new totals.

    Steps:
      1. Join lookup onto df on geo_code == LAD11CD
      2. Group by LAD21CD, summing all raw count columns
      3. Drop old pct_ columns and recompute from summed counts
      4. Rename geo_code to the 2021 code, update area_name to 2021 name
    """
    print(f"\n  [{label}]")
    before = len(df)

    lu = lookup[["LAD11CD", "LAD21CD", "LAD21NM"]].copy()

    df_joined = df.merge(lu, left_on="geo_code", right_on="LAD11CD", how="left")

    unmatched = df_joined[df_joined["LAD21CD"].isnull()]["geo_code"].tolist()
    if unmatched:
        print(f"    WARNING: {len(unmatched)} LADs not found in lookup: {unmatched[:5]}")

    df_joined = df_joined[df_joined["LAD21CD"].notna()].copy()

    exclude   = {"geo_code", "area_name", "LAD11CD", "LAD21CD", "LAD21NM", "CHGIND"}
    pct_cols  = [c for c in df_joined.columns if c.startswith("pct_")]
    count_cols = [c for c in df_joined.columns
                  if c not in exclude
                  and c not in pct_cols
                  and pd.api.types.is_numeric_dtype(df_joined[c])]

    agg_dict = {col: "sum" for col in count_cols}
    agg_dict["LAD21NM"] = "first"
    df_agg = df_joined.groupby("LAD21CD", as_index=False).agg(agg_dict)
    df_agg = df_agg.rename(columns={"LAD21CD": "geo_code", "LAD21NM": "area_name"})

    if total_col in df_agg.columns:
        df_agg = compute_percentages(df_agg, total_col)

    after = len(df_agg)
    n_merged = before - after
    print(f"    {before} 2011 LADs → {after} 2021 LADs ({n_merged} merged)")
    return df_agg


print("\n>>> APPLYING ONS BOUNDARY HARMONISATION TO ALL 2011 TABLES")
t11  = harmonise_boundaries(t11,  lookup_raw, total_col="total_households", label="QS405EW")
o11  = harmonise_boundaries(o11,  lookup_raw, total_col="total_households", label="QS412EW")
d11  = harmonise_boundaries(d11,  lookup_raw, total_col="total_households", label="QS119EW")
e11  = harmonise_boundaries(e11,  lookup_raw, total_col="total_16_74",      label="KS601EW")

print(f"\n  Post-harmonisation LAD counts:")
print(f"    t11  : {len(t11)}")
print(f"    o11  : {len(o11)}")
print(f"    d11  : {len(d11)}")
print(f"    e11  : {len(e11)}")



>>> APPLYING ONS BOUNDARY HARMONISATION TO ALL 2011 TABLES

  [QS405EW]
    331 2011 LADs → 317 2021 LADs (14 merged)

  [QS412EW]
    331 2011 LADs → 317 2021 LADs (14 merged)

  [QS119EW]
    331 2011 LADs → 317 2021 LADs (14 merged)

  [KS601EW]
    331 2011 LADs → 317 2021 LADs (14 merged)

  Post-harmonisation LAD counts:
    t11  : 317
    o11  : 317
    d11  : 317
    e11  : 317


FIND COMMON LADs ACROSS ALL TABLES

In [38]:
print("\n\n>>> FINDING COMMON LADs ACROSS ALL 6 TABLES")
sets = {
    "QS405EW": set(t11["geo_code"]),
    "QS412EW": set(o11["geo_code"]),
    "QS119EW": set(d11["geo_code"]),
    "KS601EW": set(e11["geo_code"]),
    "TS054":   set(t21["geo_code"]),
    "TS011":   set(d21["geo_code"]),
}

common_lads = set.intersection(*sets.values())
print(f"  Common LADs across all tables: {len(common_lads)}")

# Report any LADs missing from specific tables
for name, s in sets.items():
    diff = common_lads - s
    if diff:
        print(f"  WARNING: {name} missing {len(diff)} LADs: {diff}")

# Filter all tables to common LADs only
t11 = t11[t11["geo_code"].isin(common_lads)].reset_index(drop=True)
o11 = o11[o11["geo_code"].isin(common_lads)].reset_index(drop=True)
d11 = d11[d11["geo_code"].isin(common_lads)].reset_index(drop=True)
e11 = e11[e11["geo_code"].isin(common_lads)].reset_index(drop=True)
t21 = t21[t21["geo_code"].isin(common_lads)].reset_index(drop=True)
d21 = d21[d21["geo_code"].isin(common_lads)].reset_index(drop=True)

print(f"  All tables filtered to {len(common_lads)} common LADs.")



>>> FINDING COMMON LADs ACROSS ALL 6 TABLES
  Common LADs across all tables: 317
  All tables filtered to 317 common LADs.


BUILD MERGED 2011 DATASET

In [39]:
# Drop junk columns from all 2011 tables first
for _df in [t11, o11, d11, e11]:
    junk = [c for c in _df.columns if c.startswith("Unnamed")]
    _df.drop(columns=junk, inplace=True)


def safe_select(df, cols):
    found = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"  WARNING: columns not found and skipped: {missing}")
    return df[found]


print("\n>>> MERGING ALL 2011 TABLES")

t11_keep = [
    "geo_code", "area_name",
    "pct_owned_total", "pct_owned_outright", "pct_owned_mortgage",
    "pct_social_rented_total", "pct_private_rented_total", "pct_rent_free",
]

o11_keep = [
    "geo_code",
    "pct_overcrowded_total", "pct_occ_minus1", "pct_occ_minus2_less",
    "pct_occ_zero", "pct_occ_plus1", "pct_occ_plus2_more",
]

d11_keep = [
    "geo_code",
    "pct_deprived_0", "pct_deprived_1", "pct_deprived_2",
    "pct_deprived_3", "pct_deprived_4",
    "pct_multi_deprived", "pct_any_deprived",
]

e11_keep = [
    "geo_code",
    "pct_unemployed", "pct_employed_total", "pct_inactive_total",
    "pct_inactive_sick", "pct_self_employed",
]

merged_2011 = (
    safe_select(t11, t11_keep)
    .merge(safe_select(o11, o11_keep), on="geo_code", how="inner")
    .merge(safe_select(d11, d11_keep), on="geo_code", how="inner")
    .merge(safe_select(e11, e11_keep), on="geo_code", how="inner")
)
merged_2011["year"] = 2011

print(f"  Merged 2011 shape: {merged_2011.shape}")
print(f"  Columns: {list(merged_2011.columns)}")



>>> MERGING ALL 2011 TABLES
  Merged 2011 shape: (317, 27)
  Columns: ['geo_code', 'area_name', 'pct_owned_total', 'pct_owned_outright', 'pct_owned_mortgage', 'pct_social_rented_total', 'pct_private_rented_total', 'pct_rent_free', 'pct_overcrowded_total', 'pct_occ_minus1', 'pct_occ_minus2_less', 'pct_occ_zero', 'pct_occ_plus1', 'pct_occ_plus2_more', 'pct_deprived_0', 'pct_deprived_1', 'pct_deprived_2', 'pct_deprived_3', 'pct_deprived_4', 'pct_multi_deprived', 'pct_any_deprived', 'pct_unemployed', 'pct_employed_total', 'pct_inactive_total', 'pct_inactive_sick', 'pct_self_employed', 'year']


BUILD MERGED 2021 DATASET

In [40]:
print("\n\n>>> MERGING 2021 TABLES")

t21_keep = ["geo_code", "area_name",
            "pct_owned_total", "pct_owned_outright",
            "pct_social_rented_total", "pct_private_rented_total",
            "pct_rent_free"]

d21_keep = ["geo_code",
            "pct_deprived_0", "pct_deprived_1", "pct_deprived_2",
            "pct_deprived_3", "pct_deprived_4",
            "pct_multi_deprived", "pct_any_deprived"]

merged_2021 = (
    safe_select(t21, t21_keep)
    .merge(safe_select(d21, d21_keep), on="geo_code", how="inner")
)

merged_2021["year"] = 2021

print(f"  Merged 2021 shape: {merged_2021.shape}")
print(f"  Columns: {list(merged_2021.columns)}")



>>> MERGING 2021 TABLES
  Merged 2021 shape: (317, 15)
  Columns: ['geo_code', 'area_name', 'pct_owned_total', 'pct_owned_outright', 'pct_social_rented_total', 'pct_private_rented_total', 'pct_rent_free', 'pct_deprived_0', 'pct_deprived_1', 'pct_deprived_2', 'pct_deprived_3', 'pct_deprived_4', 'pct_multi_deprived', 'pct_any_deprived', 'year']


BUILD COMBINED LONG-FORMAT DATASET

In [41]:
print("\n\n>>> BUILDING COMBINED LONG-FORMAT DATASET")

# Identify columns present in both years for stacking
common_cols = (["geo_code", "area_name", "year"] +
               [c for c in merged_2011.columns
                if c in merged_2021.columns
                and c not in ("geo_code", "area_name", "year")])

combined = pd.concat(
    [merged_2011[common_cols], merged_2021[common_cols]],
    ignore_index=True
).sort_values(["geo_code", "year"])

print(f"  Combined long-format shape: {combined.shape}")
print(f"  Years present: {combined['year'].unique()}")



>>> BUILDING COMBINED LONG-FORMAT DATASET
  Combined long-format shape: (634, 15)
  Years present: [2011 2021]


In [45]:
#FINAL MISSING VALUE CHECK & REPORT

print("\n\n>>> FINAL MISSING VALUE REPORT")
for name, df in [("merged_2011", merged_2011),
                 ("merged_2021", merged_2021),
                 ("combined",    combined)]:
    total_missing = df.isnull().sum().sum()
    print(f"  {name}: {total_missing} missing values")
    if total_missing > 0:
        print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n\n>>> SAVING CLEANED FILES")

from google.colab import files
import io

def save_and_download(df, filename):
    buffer = io.BytesIO()
    df.to_csv(buffer, index=False)
    buffer.seek(0)
    with open(filename, 'wb') as f:
        f.write(buffer.read())
    files.download(filename)

save_and_download(t11,         "2011_tenure_clean.csv")
save_and_download(o11,         "2011_occupancy_clean.csv")
save_and_download(d11,         "2011_deprivation_clean.csv")
save_and_download(e11,         "2011_economic_clean.csv")
save_and_download(t21,         "2021_tenure_clean.csv")
save_and_download(d21,         "2021_deprivation_clean.csv")
save_and_download(merged_2011, "merged_2011.csv")
save_and_download(merged_2021, "merged_2021.csv")
save_and_download(combined,    "combined_2011_2021.csv")

print("All files downloaded.")
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)



>>> FINAL MISSING VALUE REPORT
  merged_2011: 0 missing values
  merged_2021: 0 missing values
  combined: 0 missing values


>>> SAVING CLEANED FILES


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All files downloaded.
PREPROCESSING COMPLETE
